# 08 - Tumor Trajectory Analysis

**Description:** Trajectory analysis of tumor cells using the STORIES (Spatio-Temporal Omics Representation and Inference of Evolutionary States) framework. This notebook fits a SpaceTime model to epithelial/malignant cells, computes potential and velocity fields, identifies driver genes, and performs transcription factor enrichment analysis.

**Conda environment:** `stories-rsc-cuda126`

## Imports

In [ ]:
import os

import anndata as ad
import jax
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import scanpy as sc
import stories

In [ ]:
jax.devices()

## Configuration

In [ ]:
DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

CHECKPOINT_DIR = os.path.join(RESULTS_DIR, "ckpt_tumor_trajectory")
TRRUST_PATH = os.path.join(DATA_DIR, "trrust_rawdata.human.tsv")

N_PCS = 20
N_NEIGHBORS = 30
QUADRATIC_WEIGHT = 1e-3
LEARNING_RATE = 1e-2
N_STEPS = 10_000

## Data Loading

In [ ]:
adata = ad.read_h5ad(os.path.join(DATA_DIR, "6-1_epithelial.h5ad"))

## PCA & Time Encoding

Select a given number of principal components then normalize the embedding.
Encode time as Normal=0, Malignant=1 using `Malignant_thr_0.50`.

In [ ]:
adata.obsm["X_pca"] = adata.obsm["X_pca"][:, :N_PCS]
adata.obsm["X_pca"] /= adata.obsm["X_pca"].max()

In [ ]:
adata.obs["time"] = adata.obs["Malignant_thr_0.50"].replace({
    "Normal": 0,
    "Malignant": 1,
})

## Spatial Coordinate Processing

Center and scale each timepoint in space. We have only one slice per time point.

In [ ]:
timepoints = np.sort(np.unique(adata.obs["time"]))
adata.obsm["spatial"] = adata.obsm["spatial"].astype(float)

for t in timepoints:
    idx = adata.obs["time"] == t
    mu = np.mean(adata.obsm["spatial"][idx, :], axis=0)
    adata.obsm["spatial"][idx, :] -= mu
    std = np.std(adata.obsm["spatial"][idx, :], axis=0)
    adata.obsm["spatial"][idx, :] /= std

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(10, 5), sharey=True, sharex=True, constrained_layout=True
)
for i, t in enumerate(sorted(adata.obs["time"].unique())):
    idx = adata.obs["time"] == t
    sc.pl.embedding(
        adata[idx], "spatial", color="StudentTVAE_pred", s=30, ax=axes[i], show=False
    )

## STORIES SpaceTime Model Fitting

In [ ]:
model = stories.SpaceTime(quadratic_weight=QUADRATIC_WEIGHT)

In [ ]:
adata.obs["time"] = adata.obs["time"].astype(int)

In [ ]:
scheduler = optax.cosine_decay_schedule(LEARNING_RATE, N_STEPS)
model.fit(
    adata=adata,
    time_key="time",
    omics_key="X_pca",
    space_key="spatial",
    optimizer=optax.adamw(scheduler),
    checkpoint_manager=CHECKPOINT_DIR,
)

## Potential Computation & Visualization

In [ ]:
stories.tools.compute_potential(adata, model, "X_pca")

In [ ]:
sc.pl.embedding(
    adata,
    basis="umap",
    color=["StudentTVAE_pred", "potential"],
    vmax="p98",
)

In [ ]:
potential_fn = lambda x: model.potential.apply(model.params, x)
df = pd.DataFrame(np.array(potential_fn(adata.obsm["X_pca"])))

df_anno = adata.obs[["StudentTVAE_pred"]]
df[["iso_1", "iso_2"]] = adata.obsm["X_umap"]
df = df.rename(columns={0: "potential"})
df = pd.concat([df, df_anno.reset_index()[["StudentTVAE_pred"]]], axis=1)
df = df.rename(columns={"StudentTVAE_pred": "annotation"})
df.to_csv(os.path.join(RESULTS_DIR, "umap_potential_tumor.csv"))

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(10, 5), sharey=True, sharex=True, constrained_layout=True
)
for i, t in enumerate(sorted(adata.obs["time"].unique())):
    idx = adata.obs["time"] == t
    sc.pl.embedding(
        adata[idx], "spatial", color="potential", s=30, ax=axes[i], show=False
    )

In [ ]:
sc.pl.embedding(
    adata, "spatial", color="potential", s=30, show=False
)

## Velocity Computation & Visualization

In [ ]:
sc.pp.neighbors(
    adata,
    n_neighbors=N_NEIGHBORS,
    n_pcs=None,
    use_rep="X_pca",
)

In [ ]:
stories.tools.compute_velocity(adata, model, "X_pca")

In [ ]:
palette = {
    "Epithelial": (0.23, 0.49, 0.77),
    "Malignant": (0.89, 0.47, 0.20),
}

stories.tools.plot_velocity(
    adata,
    "X_pca",
    basis="umap",
    color="StudentTVAE_pred",
    palette=palette,
    s=50,
)

## Driver Gene Analysis

In [ ]:
adata_stories = adata.copy()

if not isinstance(adata_stories.X, np.ndarray):
    adata_stories.X = adata_stories.X.toarray()

In [ ]:
stories.tools.regress_genes(adata_stories)

In [ ]:
sorted_genes_names = stories.tools.select_driver_genes(adata_stories, n_stages=10, n_genes=3)
fig, axes = stories.tools.plot_gene_trends(
    adata_stories, sorted_genes_names, title="NR patients"
)

In [ ]:
stories.tools.plot_single_gene_trend(
    adata_stories,
    "LAMC2",
    s=5,
    alpha=0.75,
    annotation_key="StudentTVAE_pred",
)

## TF Enrichment

In [ ]:
stories.tools.tf_enrich(
    adata_stories, trrust_path=TRRUST_PATH,
)